# 03 Hnc Upper Limits


In [ ]:
from pathlib import Path

def find_project_root(start=None):
    """Locate the project root from the working directory or an ancestor."""
    location = Path(start or Path.cwd()).resolve()
    for candidate in (location, *location.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/ directory. The README.md and notebooks/ folder must exist.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
SPECTRA_DIR = PROJECT_ROOT / "spectra"
FIGURES_DIR = PROJECT_ROOT / "figures"
SAVE_FIGURES = False  # Figures render inline; optional exports disabled by default.
SAVE_TABLES = True    # Intermediate tables are required by downstream notebooks.
# Output folders are part of the repository layout; raw data remains private.


> **Historical result:** The original notebook saved a calculation for the *southern* pointing only. The original console output is preserved below; it is not a newly calculated result. To obtain the northern result, its spectrum must be supplied and the calculation rerun.


# HNC non-detection upper-limit calculations

Methodology based on Domínguez-Gómez et al. (2022).

In [3]:
import numpy as np
import pandas as pd

# ============================================================
# CARGA DE ARCHIVOS DE HCO+ Y HCN PARA OBTENER FWHM DE REFERENCIA
# ============================================================

file_hcop = str(PROJECT_ROOT / 'results/ResultadosCLASS/W_HCO_fit_results.dat')
file_hcn  = str(PROJECT_ROOT / 'results/ResultadosCLASS/W_HCN_fit_results.dat')

df_hcop = pd.read_csv(file_hcop, sep='\s+', comment="#")
df_hcn  = pd.read_csv(file_hcn,  sep='\s+', comment="#")

# Seleccionamos SOLO las columnas necesarias
df_hcop = df_hcop[["fuente", "FWHM", "err_FWHM"]].rename(columns={
    "FWHM": "fwhm_hcop",
    "err_FWHM": "err_hcop"
})

df_hcn = df_hcn[["fuente", "FWHM", "err_FWHM"]].rename(columns={
    "FWHM": "fwhm_hcn",
    "err_FWHM": "err_hcn"
})

df_fwhm = pd.merge(df_hcop, df_hcn, on="fuente", how="inner")

if len(df_fwhm) == 0:
    raise ValueError(
        "No hay fuentes comunes entre los archivos de HCO+ y HCN. "
        "Revisa que la columna 'fuente' tenga exactamente los mismos nombres."
    )


# =============================================================
# CARGAMOS ESPECTROS DE HNC PARA CADA FUENTE
# =============================================================

source_name = "NGC3110_SOU"                  # nombre de la fuente tal como aparece en la tabla de FWHM
spectrum_file = str(PROJECT_ROOT / 'spectra/W_SHNC_spectrum.dat')        # archivo del espectro HNC para esa fuente


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def mean_FWHM(fwhm_hcop, err_hcop, fwhm_hcn, err_hcn):
    """ 
    Calcula el promedio de FWHM  de HCO+ y HCN, y su error combinado.
    """

    Delta_v = abs(fwhm_hcop + fwhm_hcn)/2
    err_Delta_v = 0.5 * np.sqrt(err_hcop**2 + err_hcn**2)

    return Delta_v, err_Delta_v


def load_spectrum(filename):
    """
    Lee un espectro con dos columnas:
    velocidad [km/s], Tmb [mK]
    """
    data = np.loadtxt(filename, skiprows=1)
    vel = data[:, 0]
    tmb = data[:, 1]
    return vel, tmb


def build_line_free_mask(vel, line_free_ranges):
    """
    Devuelve una máscara booleana que selecciona los canales
    situados dentro de los intervalos libres de línea.
    """
    mask = np.zeros(len(vel), dtype=bool)
    for vmin, vmax in line_free_ranges:
        mask |= (vel >= vmin) & (vel <= vmax)
    return mask


def compute_rms_from_line_free_channels(vel, tmb, line_free_ranges):
    """
    Calcula el rms del espectro usando únicamente las regiones
    libres de línea indicadas.
    """
    mask = build_line_free_mask(vel, line_free_ranges)

    if np.sum(mask) < 2:
        raise ValueError(
            "La máscara de regiones libres de línea selecciona menos de 2 canales. "
            "Revisa 'line_free_ranges'."
        )

    rms = np.std(tmb[mask], ddof=1)
    return rms, mask


def estimate_channel_width(vel):
    """
    Estima el ancho de canal a partir de las diferencias entre canales
    consecutivos en velocidad.
    """
    dv = np.diff(vel)

    if len(dv) == 0:
        raise ValueError("El espectro tiene menos de 2 canales.")

    # Usamos la mediana por robustez
    delta_v = np.median(np.abs(dv))
    return delta_v


def upper_limit_hnc_Kkms(rms_mK, delta_v, Delta_v):
    """
    Cota superior pedida:
        I_ul = 3 * rms * \sqrt{delta_v * Delta_v}

    Entrada:
    - rms_mK   : rms del espectro [mK]
    - Delta_v   : promedio de FWHM de HCO+ y HCN [km/s]
    - delta_v   : ancho de canal [km/s]
    Salida:
    - I_ul_Kkms : cota superior [K km/s]
    """
    I_ul_mKkms = 3.0 * rms_mK * np.sqrt(delta_v * Delta_v)
    I_ul_Kkms = I_ul_mKkms / 1000.0
    return I_ul_Kkms


def KtoJy(data, Beff, Feff, STa):
    """
    Convierte un límite de intensidad de K km/s a Jy/beam km/s
    usando el factor de conversión específico del telescopio.
    """
    return data * STa * Beff / Feff 


# ============================================================
# OBTENCIÓN DE FWHM DE REFERENCIA PARA LA FUENTE
# ============================================================

resultados2 = []

for _, row in df_fwhm.iterrows():
    Delta_v, err_Delta_v = mean_FWHM(row["fwhm_hcop"], row["err_hcop"], row["fwhm_hcn"], row["err_hcn"])

    resultados2.append({
        "fuente": row["fuente"],
        "fwhm_hcop": row["fwhm_hcop"],
        "err_hcop": row["err_hcop"],
        "fwhm_hcn": row["fwhm_hcn"],
        "err_hcn": row["err_hcn"],
        "Delta_v": Delta_v,
        "err_Delta_v": err_Delta_v,
    })

df_ref2 = pd.DataFrame(resultados2)


# ============================================================
# CARGAMOS EL ESPECTRO Y CALCULAMOS LA COTA SUPERIOR
# ============================================================

vel, tmb = load_spectrum(spectrum_file)
line_free_ranges = [(-5000,-3000), (-1000, 1000)]
rms_hnc_mK, mask_line_free = compute_rms_from_line_free_channels(vel, tmb, line_free_ranges)
delta_v = estimate_channel_width(vel)
row_ref = df_ref2[df_ref2["fuente"] == source_name].iloc[0]
Delta_v = row_ref["Delta_v"]
err_Delta_v = row_ref["err_Delta_v"]

I_ul_hnc_Kkms = upper_limit_hnc_Kkms(rms_hnc_mK, delta_v, Delta_v)
I_ul_hnc_Jy = KtoJy(I_ul_hnc_Kkms, 0.81, 0.95, 5.9)

print("=" * 80)
print(f"CÁLCULO DE LA COTA SUPERIOR DE HNC PARA LA FUENTE: {source_name}")
print("=" * 80)
print(f"Archivo de espectro       : {spectrum_file}")
print(f"FWHM medio HCO+/HCN       : {Delta_v:.2f} ± {err_Delta_v:.2f} km/s")
print(f"RMS(HNC)                  : {rms_hnc_mK:.3f} mK")
print(f"Ancho de canal            : {delta_v:.3f} km/s")
print(f"Cota superior HNC         : {I_ul_hnc_Kkms:.5f} K km/s")
print(f"Cota superior HNC         : {I_ul_hnc_Jy:.5f} Jy km/s")
print("=" * 80)


# ============================================================
# GUARDAR RESULTADOS EN ARCHIVO
# ============================================================

output_file = str(RESULTS_DIR / "ResultadosHNC" / "upper_limit_HNC_2_results.dat")

if SAVE_TABLES:
    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "a") as f:
        # Escribimos cabecera solo si el archivo no existe o está vacío
        if f.tell() == 0:
            f.write("fuente\trms_mK\tDelta_v_kms\tdelta_v_kms\tI_ul_Jy\tI_ul_Kkms\n")

        f.write(f"{source_name}\t{rms_hnc_mK:.5f}\t{Delta_v:.2f}\t{delta_v:.2f}\t{I_ul_hnc_Jy:.6f}\t{I_ul_hnc_Kkms:.5f}\n")

    print(f"Resultado guardado en: {output_file}")


from IPython.display import display
upper_limit_summary = pd.DataFrame([{
    "Source": source_name,
    "RMS (mK)": rms_hnc_mK,
    "Reference FWHM (km/s)": Delta_v,
    "Channel width (km/s)": delta_v,
    "HNC 3-sigma upper limit (K km/s)": I_ul_hnc_Kkms,
    "HNC 3-sigma upper limit (Jy km/s)": I_ul_hnc_Jy,
}])
display(upper_limit_summary)


CÁLCULO DE LA COTA SUPERIOR DE HNC PARA LA FUENTE: NGC3110_SOU
Archivo de espectro       : <PRIVATE_PROJECT_ROOT>/spectra/W_SHNC_spectrum.dat
FWHM medio HCO+/HCN       : 277.70 ± 38.22 km/s
RMS(HNC)                  : 0.626 mK
Ancho de canal            : 27.131 km/s
Cota superior HNC         : 0.16314 K km/s
Cota superior HNC         : 0.82066 Jy km/s
Resultado guardado en: <PRIVATE_PROJECT_ROOT>/results/ResultadosHNC/upper_limit_HNC_2_results.dat
